## 📊 Análisis de Ventas en el rubro Farmacéutico 
### Proyecto Final – Data Science

Autor/a: Rubis Becerra

### Abstract

El presente proyecto tiene como objetivo analizar el comportamiento de las ventas de una empresa a partir de un conjunto de datos compuesto por 5.000 registros y 19 variables, que incluyen información sobre distribuidores, clientes, ubicación geográfica, canales de venta, productos, equipos comerciales y métrica de desempeño. El dataset combina variables categóricas y numéricas, lo que permite realizar un análisis exploratorio de datos (EDA) integral, para comprender los factores que influyen en el comportamiento de la variable sales, la cual representa el ingreso generado por cada transacción.

Durante esta primera etapa del proyecto se realizó un proceso de limpieza y preparación de los datos, identificando valores faltantes en variables clave como ventas, cantidad, ubicación y personal comercial. El análisis exploratorio se apoyó en resúmenes estadísticos y visualizaciones univariadas y bivariadas, incluyendo gráficos de barras, histogramas, boxplot, diagrama de dispersión y matriz de correlación con escala logarítmica.

En una primera instancia, el análisis se enfoca en entender la estructura general del negocio mediante la clasificación de las ventas en niveles (Low, Medium, High). Esto permite evaluar la distribución del ingreso y determinar si el negocio depende principalmente de transacciones de bajo ticket o si existe una concentración relevante en ventas de alto valor.

Posteriormente, se analiza el comportamiento de las ventas en función del país siendo una variable categórica estratégica, tales como región para identificar posibles diferencias estructurales entre segmentos. Este enfoque permite detectar oportunidades comerciales y áreas de mejora.

Asimismo, se examina la relación entre sales y variables numéricas clave como quantity y price, dado que estas representan componentes directos en la generación del ingreso. Se evalúa la existencia de relaciones lineales o patrones de dispersión que puedan aportar capacidad predictiva al modelo.

Los hallazgos obtenidos en este EDA permitirán seleccionar las variables más relevantes y establecer fundamentos sólidos para el posterior desarrollo del modelo de predicción de ventas.


### Pregunta Problema

¿Qué variables explican de manera significativa el comportamiento de las ventas (sales) y cómo pueden utilizarse para construir un modelo predictivo confiable?

### Preguntas e Hipótesis de Interés

- ¿Cuál es la distribución de valores en ventas (sales) y su impacto?
- ¿Cómo se distribuyen los niveles de ventas?, ¿Cúal es el nivel de ventas predominante según el monto acumulado?
- ¿Cómo influye la región en la cantidad y magnitud de transacciones?
- ¿Cuál es la relación entre ventas con la cantidad y el precio?

Hipótesis iniciales:
- Es importante conocer la distribución de los valores de ventas ya que puede definir las estructuras del modelo predictivo.
- La mayoría de las transacciones se concentran en niveles bajos de ventas, lo que indica una posible dependencia del negocio en operaciones de menor ticket.
- Las ventas presentan diferencias significativas según variables claves del negocio (por ejemplo la región).
- Existe una relación positiva entre las ventas, cantidad y precio, lo que sugiere que estas variables son fuertes candidatas para el modelo predictivo.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/pharma_rsnx.csv")

In [3]:
df.head(3)

,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team,Sales_Level
0,Prohaska-Kuhic,"Wunsch, Mills and Walter",Kielce,Poland,50.8725,NaN,Pharmacy,Institution,Tacrodomide,Antipiretics,10.0,420,4200.0,January,2018,Stella Given,Alisha Cordwell,Charlie,Low
1,Cassin,Block-Romaguera Pharmaceutical Limited,Tarnowskie Góry,Poland,50.4500,18.8667,Hospital,Private,Finanel,Antimalarial,20.0,206,4120.0,January,2018,Sheila Stones,Britanny Bold,Delta,Low
2,Smith Inc,Doyle-Tillman Pharmaceutical Ltd,Dęblin,Poland,51.5667,21.8614,Hospital,Private,Tacrodomide,Antipiretics,2.0,420,840.0,January,2018,Stella Given,Alisha Cordwell,Charlie,Low


In [4]:
df.shape

(5000, 19)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Distributor        5000 non-null   object 
 1   Customer Name      5000 non-null   object 
 2   City               5000 non-null   object 
 3   Country            5000 non-null   object 
 4   Latitude           4750 non-null   float64
 5   Longitude          4750 non-null   float64
 6   Channel            5000 non-null   object 
 7   Sub-channel        5000 non-null   object 
 8   Product Name       5000 non-null   object 
 9   Product Class      5000 non-null   object 
 10  Quantity           4550 non-null   float64
 11  Price              5000 non-null   int64  
 12  Sales              4400 non-null   float64
 13  Month              5000 non-null   object 
 14  Year               5000 non-null   int64  
 15  Name of Sales Rep  4750 non-null   object 
 16  Manager            4550 

In [6]:
df.columns

Index(['Distributor', 'Customer Name', 'City', 'Country', 'Latitude',
       'Longitude', 'Channel', 'Sub-channel', 'Product Name', 'Product Class',
       'Quantity', 'Price', 'Sales', 'Month', 'Year', 'Name of Sales Rep',
       'Manager', 'Sales Team', 'Sales_Level'],
      dtype='object')

In [7]:
df.isna().sum()

Distributor            0
Customer Name          0
City                   0
Country                0
Latitude             250
Longitude            250
Channel                0
Sub-channel            0
Product Name           0
Product Class          0
Quantity             450
Price                  0
Sales                600
Month                  0
Year                   0
Name of Sales Rep    250
Manager              450
Sales Team             0
Sales_Level            0
dtype: int64

In [8]:
#Se tratan los valores nulos de latitude y longitude imputando con la media de esos valores por ciudad
def imputar_media(x):
    return x.fillna(x.mean())

df[['Latitude', 'Longitude']] = (
    df.groupby('City')[['Latitude', 'Longitude']]
      .transform(imputar_media)
)

In [9]:
#Se imputan los valores nulos de quantity con la mediana de la columna. 
#Se usa .loc y evitó inplace=True para prevenir problemas de chained assignment y asegurar compatibilidad con versiones futuras de pandas.

df.loc[:, 'Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

In [10]:
#Se tratan los valores nulos de Sales recalculando dicha variable.
df['Sales'] = df['Quantity'] * df['Price']

In [11]:
#Se tratan los valores nulos de Name of Sales Rep y Manager rellenando con la moda para ambas variables.
def imputar_moda(x):
    return x.fillna(x.mode()[0])

df['Name of Sales Rep'] = (
    df.groupby('Sales Team')['Name of Sales Rep']
      .transform(imputar_moda)
)

df['Manager'] = (
    df.groupby('Sales Team')['Manager']
      .transform(imputar_moda)
)

In [12]:
#Se verifica nuevamente que no hayan quedado valores nulos.
df.isna().sum()

Distributor          0
Customer Name        0
City                 0
Country              0
Latitude             0
Longitude            1
Channel              0
Sub-channel          0
Product Name         0
Product Class        0
Quantity             0
Price                0
Sales                0
Month                0
Year                 0
Name of Sales Rep    0
Manager              0
Sales Team           0
Sales_Level          0
dtype: int64

In [13]:
#Se tratan los valores nulos que quedan para la variable longitude con la media de esos valores por país 
def imputar_media(x):
    return x.fillna(x.mean())

df[['Longitude']] = (
    df.groupby('Country')[['Longitude']]
      .transform(imputar_media)
)

In [14]:
#Se verifica.
df.isna().sum()

Distributor          0
Customer Name        0
City                 0
Country              0
Latitude             0
Longitude            0
Channel              0
Sub-channel          0
Product Name         0
Product Class        0
Quantity             0
Price                0
Sales                0
Month                0
Year                 0
Name of Sales Rep    0
Manager              0
Sales Team           0
Sales_Level          0
dtype: int64

In [15]:
df.Distributor.unique()

array(['Prohaska-Kuhic ', 'Cassin  ', 'Smith Inc ', 'Carter-Conn  ',
       'Gottlieb-Cruickshank  ', 'Beier  ', 'Schuppe Inc ', 'Rohan   ',
       'Graham and Sons ', 'Stehr-Champlin  ', 'Kris LLC  ',
       'Lindgren-Simonis Pharm', 'Rogahn-Klein ', 'Gerlach LLC ',
       'Gleason   ', 'Erdman  ', 'Daugherty-Rempel  ', 'Koss   ',
       'Crist Inc ', 'Lockman  ', 'Romaguera-Fay  ', 'Schaefer LLC ',
       'Rohan and Sons  ', 'Welch-Langworth ', 'Kozey-Emmerich ',
       'Lesch   ', 'Hansen Group Pharm', 'Nader-Gaylord  '], dtype=object)

In [16]:
df.Country.unique()

array(['Poland', 'Germany'], dtype=object)

In [17]:
df.Channel.unique()

array(['Pharmacy', 'Hospital'], dtype=object)

In [18]:
df["Product Class"].unique()

array(['Antipiretics', 'Antimalarial', 'Analgesics', 'Mood Stabilizers',
       'Antiseptics', 'Antibiotics'], dtype=object)

In [19]:
df.Manager.unique()

array(['Alisha Cordwell', 'Britanny Bold', 'Tracy Banks',
       'James Goodwill'], dtype=object)

In [20]:
df["Sales Team"].unique()

array(['Charlie', 'Delta', 'Bravo', 'Alfa'], dtype=object)

In [21]:
df.Sales_Level.unique()

array(['Low', 'Medium', 'High'], dtype=object)

In [22]:
df.to_csv("../data/processed/pharma_rsnx_processed.csv", index=False)